In [68]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import tensorflow_recommenders as tfrs

In [69]:
ratings = tfds.load("movielens/100k-ratings", split="train")

ratings = ratings.map(lambda x: {
    "user_id": x["user_id"],
    "movie_title": x["movie_title"],
    "user_rating": x["user_rating"]
})

In [70]:
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [71]:
movie_titles = ratings.batch(1_000_000).map(lambda x: x["movie_title"])
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

In [72]:
print(len(ratings))
print(len(unique_user_ids))
print(len(unique_movie_titles))

100000
943
1664


- 랭킹 모델은 검색모델과 동일한 효율성 제약에 직면하지 않으므로 아키텍처선택에 있어 조금 더 자유로워진다.

In [73]:
class RankingModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        embedding_dimension = 32
        
        self.user_vocabulary = unique_user_ids
        self.movie_vocabulary = unique_movie_titles
        
        
        self.user_embeddings = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=self.user_vocabulary, mask_token=None),
            tf.keras.layers.Embedding(len(self.user_vocabulary)+1, embedding_dimension)
        ])
        
        self.movie_embeddings = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=self.movie_vocabulary, mask_token=None),
            tf.keras.layers.Embedding(len(self.movie_vocabulary)+1, embedding_dimension)
        ])
        
        self.ratings = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation="relu"),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(1)
        ])
        
    def call(self, inputs):
        user_id, movie_title = inputs
        
        user_embedding = self.user_embeddings(user_id)
        movie_embedding = self.movie_embeddings(movie_title)
        
        return self.ratings(tf.concat([user_embedding, movie_embedding], axis=1))

In [74]:
RankingModel()((["42"], ["One Flew Over the Cuckoo's Nest (1975)"])) # 튜플 형태() 안에 []이거는 데이터상자array

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[0.00081685]], dtype=float32)>

In [75]:
task = tfrs.tasks.Ranking(
    loss = tf.keras.losses.MeanSquaredError(),
    metrics = [tf.keras.metrics.RootMeanSquaredError()]
)

In [76]:
# 전체 모델
class MovielensModel(tfrs.models.Model):

  def __init__(self):
    super().__init__()
    self.ranking_model: tf.keras.Model = RankingModel()
    self.task: tf.keras.layers.Layer = tfrs.tasks.Ranking(
      loss = tf.keras.losses.MeanSquaredError(),
      metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

  def call(self, features: Dict[str, tf.Tensor]) -> tf.Tensor:
    return self.ranking_model(
        (features["user_id"], features["movie_title"]))

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    labels = features.pop("user_rating")

    rating_predictions = self(features)

    # The task computes the loss and the metrics.
    return self.task(labels=labels, predictions=rating_predictions)

class실행이 compute_loss가 먼저 실행됨. 그다음 call

In [77]:
model = MovielensModel()
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [78]:
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()

In [79]:
model.fit(cached_train, epochs=3)

Epoch 1/3


2026-03-14 21:17:50.337227: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adagrad/AssignAddVariableOp.


10/10 [==============================] - 2s 49ms/step - root_mean_squared_error: 1.7767 - loss: 2.9471 - regularization_loss: 0.0000e+00 - total_loss: 2.9471
Epoch 2/3
10/10 [==============================] - 0s 10ms/step - root_mean_squared_error: 1.1236 - loss: 1.2651 - regularization_loss: 0.0000e+00 - total_loss: 1.2651
Epoch 3/3
10/10 [==============================] - 0s 11ms/step - root_mean_squared_error: 1.1218 - loss: 1.2610 - regularization_loss: 0.0000e+00 - total_loss: 1.2610


In [80]:
model.evaluate(cached_test, return_dict=True)

5/5 [==============================] - 1s 12ms/step - root_mean_squared_error: 1.1196 - loss: 1.2481 - regularization_loss: 0.0000e+00 - total_loss: 1.2481


{'root_mean_squared_error': 1.1195613145828247,
 'loss': 1.224717617034912,
 'regularization_loss': 0.0,
 'total_loss': 1.224717617034912}

In [81]:
test_ratings = {}
test_movie_titles = ["M*A*S*H (1970)", "Dances with Wolves (1990)", "Speed (1994)"]

for movie_title in test_movie_titles:
    test_ratings[movie_title] = model({
        "user_id": np.array(["42"]),
        "movie_title": np.array([movie_title])
    })
    
print("평점:")
for title, score in sorted(test_ratings.items(), key=lambda x: x[1], reverse=True):
    print(f"{title}: {score}")
    

평점:
Speed (1994): [[1.4499755]]
Dances with Wolves (1990): [[1.4361]]
M*A*S*H (1970): [[1.430429]]


In [85]:
model.save_weights("my_ranking_weights.weights.h5")

In [ ]:
new_model = MovielensModel()

# 가짜 데이터를 한 번 흘려보내서 모델을 깨워양함. (한번은무조건흘러가야됨)
_ = new_model({
    "user_id": np.array(["42"]), 
    "movie_title": np.array(["Speed (1994)"])
})

In [97]:
new_model.load_weights("my_ranking_weights.weights.h5")

In [98]:
new_model({
    "user_id": np.array(["42"]), 
    "movie_title": np.array(["Speed (1994)"])
}).numpy()

array([[1.4499755]], dtype=float32)

In [103]:
model({
    "user_id": np.array(["42"]), 
    "movie_title": np.array(["Speed (1994)"])
}).numpy()

array([[1.4499755]], dtype=float32)

디버깅노트: 모델 저장이 아닌 가중치 저장을해 로드했더니 해결. + unique_user_ids등을 모델안에서 self.로 명시함.